# 01 — Raw folders to per-dataset Stage 1 HDF5

This notebook is the ingestion entry point. It discovers raw files, joins patient labels, previews the actual parser output, builds validated manifests, and writes one or more nested cell-count levels to per-dataset Stage 1 files under `data/stage1/`. All three supported datasets (BLAST110, LAIP29, FLOWCAPII) are processed independently in the same run; a failure in one dataset is reported and does not block the others.

## What is read

- **FCS (`.fcs`)**: read with FlowIO. Events become `[cells, channels]`; marker names use FCS `PnS` and fall back to `PnN`. Install with `pip install -e '.[fcs,notebook]'`. Compensation and biological transforms are intentionally **not** guessed here; apply them explicitly during preprocessing.
- **CSV/TSV/TXT**: rows are cells and columns are markers. A text header supplies marker names. Headerless numeric files need `markers_by_tube`.
- **NPY**: a 2-D numeric `[cells, markers]` array. Supply `markers_by_tube` for meaningful names.
- **NPZ**: use key `cells` (or the first array); an optional `markers` array supplies names.

Labels are never inferred from FCS metadata. They are joined from an explicit sample-information table and checked for consistency across tubes. `counts` records the original number of events before subsampling.

## Expected input layout

The raw tree (`RAW_INPUT_ROOT`) is located automatically by walking up from this repository, and every generated output is written relative to the repository root:

```text
<repo root>/
└── data/
    ├── manifests/                        # generated here (one CSV per dataset)
    │   ├── blast110.csv
    │   ├── laip29.csv
    │   └── flowcapii.csv
    └── stage1/                           # generated here (one HDF5 per dataset)
        ├── BLAST110_stage1.h5
        ├── LAIP29_stage1.h5
        └── FLOWCAPII_stage1.h5

RAW_INPUT_ROOT/
├── BLAST110/{FCS/, labels/, sample_info.csv}
├── LAIP29/{FCS/, labels/, annotations/, sample_info.csv}
└── FlowCAPII/{FCS/, attachments/AML.csv}
```

BLAST110 and LAIP29 are the public cMRD layouts. FlowCAPII is the third dataset used by the legacy FlowCode archive.

In [1]:
%pip install -e '../.[fcs]'

Obtaining file:///home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for flowlot (pyproject.toml) ... done
  Created wheel for flowlot: filename=flowlot-0.1.0-0.editable-py3-none-any.whl size=5485 sha256=5f395505ee451a4139075f81bb67d35c926bd5d679430bebb50fb9f5f282ee8d
  Stored in directory: /tmp/pip-ephem-wheel-cache-tt_lsi78/wheels/fa/03/d4/01b20403a127c3b393d15823372a6f9b251a12835156438ba1
Successfully built flowlot
  Attempting uninstall: flowlot
    Found existing installation: flowlot 0.1.0
    Uninstalling flowlot-0.1.0:
      Successfully uninstalled flowlot-0.1.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
from pathlib import Path
import re
import time

import h5py
import numpy as np
import pandas as pd

from flowlot.io import (
    audit_manifest,
    audit_stage1,
    build_stage1_from_manifest,
    create_manifest_from_folder,
    create_manifest_from_metadata,
    load_cytometry_file,
)
from flowlot.io.manifest import SUPPORTED_INPUTS, is_ignored_input

## 1. Configure paths and all three datasets

Every later cell loops over `DATASETS` — BLAST110, LAIP29, and FlowCAPII — independently. A failure in one dataset is reported and recorded in the final summary without blocking the remaining datasets. Directories are derived from this notebook's own location with `pathlib`, so moving or copying the project keeps every output path valid.

In [3]:
# -------------------- PATH SETUP (relative to this notebook; the project is portable) --------------------
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()                # .../P1_FlowLOT_opencode/notebooks
PROJECT_ROOT = NOTEBOOK_DIR.parent       # .../P1_FlowLOT_opencode
DATA_ROOT = PROJECT_ROOT / 'data'        # every generated output lives under ../data/
MANIFEST_DIR = DATA_ROOT / 'manifests'   # per-dataset ingestion manifests
STAGE1_DIR = DATA_ROOT / 'stage1'        # per-dataset Stage 1 HDF5 files
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_DIR.mkdir(parents=True, exist_ok=True)
print('Working directory :', NOTEBOOK_DIR.resolve())
print('Project root      :', PROJECT_ROOT.resolve())
print('Output root       :', DATA_ROOT.resolve())


def _locate_raw_root() -> Path:
    """Locate the raw cytometry tree without hard-coding any absolute path.

    Walk up from the repository root looking for a sibling `FlowLOT_main/Data/
    Raw_Data` tree, or fall back on an in-repository `raw_cytometry_input`
    mirror if present.
    """
    mirror = PROJECT_ROOT / 'raw_cytometry_input'
    if mirror.is_dir():
        return mirror
    for ancestor in (PROJECT_ROOT, *PROJECT_ROOT.parents):
        candidate = ancestor / 'FlowLOT_main' / 'Data' / 'Raw_Data'
        if candidate.is_dir():
            return candidate
    return PROJECT_ROOT.parent / 'FlowLOT_main' / 'Data' / 'Raw_Data'


RAW_INPUT_ROOT = Path(_locate_raw_root()).resolve()
print('Raw input root    :', RAW_INPUT_ROOT)
print('Raw input exists  :', RAW_INPUT_ROOT.is_dir())
if not RAW_INPUT_ROOT.is_dir():
    print('WARNING: raw input root not found; affected datasets will be skipped.')

# Global ingestion switches
CELL_COUNTS = [1000]                    # subsampled cells per patient/tube written to Stage 1
SEED = 42
RECURSIVE = True
STRICT_FILENAMES = True
GENERATE_MANIFESTS = True               # keep False to inspect discovery without writing
OVERWRITE_MANIFESTS = True
RUN_BUILD = True                        # keep False to audit/preview before building

# All supported datasets. Each dataset gets its own manifest and Stage 1 file
# under ../data so a single failure is isolated and reported.
DATASET_PRESETS = {
    'BLAST110': {
        'dataset': 'BLAST110',
        'manifest_mode': 'folder',
        'raw_root': RAW_INPUT_ROOT / 'BLAST110' / 'FCS',
        'labels_csv': RAW_INPUT_ROOT / 'BLAST110' / 'sample_info.csv',
        'manifest': MANIFEST_DIR / 'blast110.csv',
        'stage1': STAGE1_DIR / 'BLAST110_stage1.h5',
        # Legacy sample_info.csv joins by full filename stem, not patient_id.
        'filename_pattern': r'BLAST110_(?P<patient_id>[0-9]+)_(?P<tube_id>P[0-9]+)\.(?:fcs|csv|tsv|txt|npy|npz)$',
        'labels_id_column': 'BLAST110_ID',
        'labels_label_column': 'sample_type',
        'labels_match': 'file_id',
        # One event-level CSV per raw file: event_ID,WBC,Blast.
        'event_labels_root': RAW_INPUT_ROOT / 'BLAST110' / 'labels',
        'event_labels_pattern': '{stem}.csv',
        'event_id_column': 'event_ID',
        'population_columns': ['WBC', 'Blast'],
        'markers_by_tube': {},
    },
    'LAIP29': {
        'dataset': 'LAIP29',
        'manifest_mode': 'folder',
        'raw_root': RAW_INPUT_ROOT / 'LAIP29' / 'FCS',
        'labels_csv': RAW_INPUT_ROOT / 'LAIP29' / 'sample_info.csv',
        'manifest': MANIFEST_DIR / 'laip29.csv',
        'stage1': STAGE1_DIR / 'LAIP29_stage1.h5',
        'filename_pattern': r'LAIP29_(?P<patient_id>[0-9]+_(?:Dx|FU))_(?P<tube_id>P[0-9]+)\.(?:fcs|csv|tsv|txt|npy|npz)$',
        'labels_id_column': 'LAIP29_ID',
        'labels_label_column': 'sample_type',
        'labels_match': 'file_id',
        'event_labels_root': RAW_INPUT_ROOT / 'LAIP29' / 'labels',
        'event_labels_pattern': '{stem}.csv',
        'event_id_column': 'event_ID',
        'population_columns': ['WBC', 'Blast', 'LAIP'],
        'markers_by_tube': {},
    },
    'FLOWCAPII': {
        'dataset': 'FLOWCAPII',
        'manifest_mode': 'metadata',
        'raw_root': RAW_INPUT_ROOT / 'FlowCAPII' / 'FCS',
        'metadata_csv': RAW_INPUT_ROOT / 'FlowCAPII' / 'attachments' / 'AML.csv',
        'manifest': MANIFEST_DIR / 'flowcapii.csv',
        'stage1': STAGE1_DIR / 'FLOWCAPII_stage1.h5',
        'file_column': 'FCS file',
        'patient_id_column': 'Individual',
        'tube_id_column': 'Tube number',
        'label_column': 'Condition',
        'tube_prefix': 'P',
        'markers_by_tube': {},
    },
}

DATASETS = list(DATASET_PRESETS.values())    # <-- every later cell loops over all three
print('Configured datasets:', [cfg['dataset'] for cfg in DATASETS])

Working directory : /home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/notebooks
Project root      : /home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode
Output root       : /home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/data
Raw input root    : /home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data
Raw input exists  : True
Configured datasets: ['BLAST110', 'LAIP29', 'FLOWCAPII']


BLAST110 and LAIP29 have two label inputs: `sample_info.csv` supplies the sample class, and one CSV per FCS file supplies event-level populations. FlowCAPII instead uses `attachments/AML.csv` for the raw filename, individual, tube, and condition and has no event-label sidecar in the legacy pipeline.

In [4]:
display(pd.DataFrame({'BLAST110_ID': ['BLAST110_1_P1'], 'sample_type': ['AML_Dx']}))
display(pd.DataFrame({'event_ID': [1, 2, 3], 'WBC': [1, 1, 0], 'Blast': [1, 0, 0]}))

,BLAST110_ID,sample_type
0,BLAST110_1_P1,AML_Dx


,event_ID,WBC,Blast
0,1,1,1
1,2,1,0
2,3,0,0


## 2. Discover files before writing anything

This dry run shows which folder is read, which files are eligible, and which names fail the patient/tube rule.

In [5]:
for cfg in DATASETS:
    root = cfg['raw_root'].resolve()
    if not root.is_dir():
        print(f"MISSING raw_root: {root}")
        continue
    iterator = root.rglob('*') if RECURSIVE else root.glob('*')
    files = sorted(
        p for p in iterator
        if p.is_file() and p.suffix.lower() in SUPPORTED_INPUTS
        and not is_ignored_input(p, root)
    )
    print(f"\n{cfg['dataset']}: reading {root}")
    print(f"supported={len(files)}, formats={dict(Counter(p.suffix.lower() for p in files))}")
    if cfg['manifest_mode'] == 'folder':
        pattern = re.compile(cfg['filename_pattern'])
        matched = [p for p in files if pattern.search(p.relative_to(root).as_posix())]
        unmatched = [p.relative_to(root).as_posix() for p in files if p not in matched]
        print(f'matched={len(matched)}')
        print('matched examples:', [p.relative_to(root).as_posix() for p in matched[:8]])
        if unmatched:
            print('UNMATCHED examples:', unmatched[:8])
    else:
        metadata_path = cfg['metadata_csv']
        print('metadata table:', metadata_path)
        if metadata_path.exists():
            metadata_preview = pd.read_csv(metadata_path)
            print('metadata columns:', metadata_preview.columns.tolist())
            display(metadata_preview.head())
        else:
            print('MISSING metadata table')


BLAST110: reading /home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/BLAST110/FCS
supported=403, formats={'.fcs': 403}
matched=403
matched examples: ['BLAST110_100_P1.fcs', 'BLAST110_100_P2.fcs', 'BLAST110_100_P3.fcs', 'BLAST110_100_P4.fcs', 'BLAST110_101_P1.fcs', 'BLAST110_101_P2.fcs', 'BLAST110_101_P3.fcs', 'BLAST110_101_P4.fcs']

LAIP29: reading /home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/LAIP29/FCS
supported=82, formats={'.fcs': 82}
matched=82
matched examples: ['LAIP29_10_Dx_P1.fcs', 'LAIP29_10_Dx_P2.fcs', 'LAIP29_11_Dx_P1.fcs', 'LAIP29_11_Dx_P2.fcs', 'LAIP29_11_FU_P1.fcs', 'LAIP29_11_FU_P2.fcs', 'LAIP29_12_Dx_P1.fcs', 'LAIP29_12_Dx_P2.fcs']

FLOWCAPII: reading /home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/FlowCAPII/FCS
supported=2872, formats={'.fcs': 2872}
metadata table: /home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/FlowCAPII/attachments/AML.csv
metadata columns: ['FCS file', 'Tube number', 'Individual', 'Condition']


,FCS file,Tube number,Individual,Condition
0,0001.FCS,1,1,normal
1,0002.FCS,2,1,normal
2,0003.FCS,3,1,normal
3,0004.FCS,4,1,normal
4,0005.FCS,5,1,normal


## 3. Generate the manifest

Set `GENERATE_MANIFESTS=True` only after discovery looks correct. Paths are stored relative to the manifest, making the project movable. Generation refuses to overwrite an existing manifest unless explicitly enabled.

In [6]:
if GENERATE_MANIFESTS:
    for cfg in DATASETS:
        if cfg['manifest_mode'] == 'folder':
            frame = create_manifest_from_folder(
                raw_root=cfg['raw_root'], output=cfg['manifest'],
                filename_pattern=cfg['filename_pattern'], labels=cfg['labels_csv'],
                markers_by_tube=cfg.get('markers_by_tube'),
                labels_id_column=cfg['labels_id_column'],
                labels_label_column=cfg['labels_label_column'],
                labels_match=cfg['labels_match'],
                event_labels_root=cfg['event_labels_root'],
                event_labels_pattern=cfg['event_labels_pattern'],
                event_id_column=cfg['event_id_column'],
                population_columns=cfg['population_columns'],
                recursive=RECURSIVE, strict=STRICT_FILENAMES,
                overwrite=OVERWRITE_MANIFESTS,
            )
        else:
            frame = create_manifest_from_metadata(
                raw_root=cfg['raw_root'], metadata=cfg['metadata_csv'],
                output=cfg['manifest'], file_column=cfg['file_column'],
                patient_id_column=cfg['patient_id_column'],
                tube_id_column=cfg['tube_id_column'], label_column=cfg['label_column'],
                tube_prefix=cfg['tube_prefix'], markers_by_tube=cfg['markers_by_tube'],
                overwrite=OVERWRITE_MANIFESTS,
            )
        print(f"wrote {cfg['manifest']} ({len(frame)} patient/tube rows)")
else:
    print('Dry run only: set GENERATE_MANIFESTS=True after reviewing discovery.')

wrote /home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/data/manifests/blast110.csv (403 patient/tube rows)
wrote /home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/data/manifests/laip29.csv (82 patient/tube rows)
wrote /home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/data/manifests/flowcapii.csv (2872 patient/tube rows)


## 4. Audit labels, tubes, paths, and marker declarations

The audit catches missing files, duplicate patient/tube rows, conflicting patient labels, non-finite values in the manifest, and invalid marker declarations.

In [7]:
for cfg in DATASETS:
    if not cfg['manifest'].exists():
        print(f"MISSING manifest: {cfg['manifest']}")
        continue
    inventory, issues = audit_manifest(cfg['manifest'])
    print(f"\n{cfg['dataset']}: rows={len(inventory)}, issues={len(issues)}")
    display(inventory.head())
    if not issues.empty:
        display(issues)


BLAST110: rows=403, issues=0


,patient_id,tube_id,path,label,markers,event_labels_path,event_id_column,population_columns,resolved_path
0,1,P1,../../../../../../FlowLOT_main/Data/Raw_Data/B...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/B...,event_ID,WBC;Blast,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
1,1,P2,../../../../../../FlowLOT_main/Data/Raw_Data/B...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/B...,event_ID,WBC;Blast,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
2,1,P3,../../../../../../FlowLOT_main/Data/Raw_Data/B...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/B...,event_ID,WBC;Blast,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
3,1,P4,../../../../../../FlowLOT_main/Data/Raw_Data/B...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/B...,event_ID,WBC;Blast,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
4,10,P1,../../../../../../FlowLOT_main/Data/Raw_Data/B...,AML_FU,,../../../../../../FlowLOT_main/Data/Raw_Data/B...,event_ID,WBC;Blast,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...



LAIP29: rows=82, issues=0


,patient_id,tube_id,path,label,markers,event_labels_path,event_id_column,population_columns,resolved_path
0,10_Dx,P1,../../../../../../FlowLOT_main/Data/Raw_Data/L...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/L...,event_ID,WBC;Blast;LAIP,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
1,10_Dx,P2,../../../../../../FlowLOT_main/Data/Raw_Data/L...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/L...,event_ID,WBC;Blast;LAIP,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
2,11_Dx,P1,../../../../../../FlowLOT_main/Data/Raw_Data/L...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/L...,event_ID,WBC;Blast;LAIP,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
3,11_Dx,P2,../../../../../../FlowLOT_main/Data/Raw_Data/L...,AML_Dx,,../../../../../../FlowLOT_main/Data/Raw_Data/L...,event_ID,WBC;Blast;LAIP,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
4,11_FU,P1,../../../../../../FlowLOT_main/Data/Raw_Data/L...,AML_FU,,../../../../../../FlowLOT_main/Data/Raw_Data/L...,event_ID,WBC;Blast;LAIP,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...



FLOWCAPII: rows=2872, issues=0


,patient_id,tube_id,path,label,markers,resolved_path
0,1,P1,../../../../../../FlowLOT_main/Data/Raw_Data/F...,normal,,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
1,1,P2,../../../../../../FlowLOT_main/Data/Raw_Data/F...,normal,,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
2,1,P3,../../../../../../FlowLOT_main/Data/Raw_Data/F...,normal,,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
3,1,P4,../../../../../../FlowLOT_main/Data/Raw_Data/F...,normal,,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...
4,1,P5,../../../../../../FlowLOT_main/Data/Raw_Data/F...,normal,,/home/qpb3vt/Drive/FlowLOT_main/Data/Raw_Data/...


## 5. Parse a few real files before building HDF5

This invokes the same raw reader as the builder and verifies matrix shape, marker names, event-level CSV columns, and original WBC/Blast/LAIP sums. During the build, rows are joined by `event_ID` before subsampling. Increase `PREVIEW_FILES` cautiously for very large FCS files.

In [8]:
PREVIEW_FILES = 3
preview_rows = []
for cfg in DATASETS:
    manifest_path = cfg['manifest'].resolve()
    if not manifest_path.exists():
        continue
    manifest = pd.read_csv(manifest_path, dtype=str).fillna('')
    for row in manifest.head(PREVIEW_FILES).itertuples(index=False):
        source = Path(row.path)
        if not source.is_absolute():
            source = manifest_path.parent / source
        declared = [m.strip() for m in row.markers.split(';') if m.strip()] or None
        try:
            cells, markers = load_cytometry_file(source, declared)
            population_summary = {}
            event_labels_value = getattr(row, 'event_labels_path', '')
            if event_labels_value:
                event_labels_path = Path(event_labels_value)
                if not event_labels_path.is_absolute():
                    event_labels_path = manifest_path.parent / event_labels_path
                event_labels = pd.read_csv(event_labels_path)
                populations = [p for p in getattr(row, 'population_columns', '').split(';') if p]
                population_summary = event_labels[populations].sum().to_dict()
            preview_rows.append({
                'dataset': cfg['dataset'], 'patient_id': row.patient_id, 'tube_id': row.tube_id,
                'file': source.name, 'shape': cells.shape, 'original_count': len(cells),
                'finite_fraction': float(np.isfinite(cells).mean()),
                'markers': markers[:8], 'original_population_counts': population_summary,
                'status': 'OK',
            })
        except Exception as exc:
            preview_rows.append({'dataset': cfg['dataset'], 'file': source.name, 'status': repr(exc)})
display(pd.DataFrame(preview_rows))

/home/qpb3vt/anaconda3/envs/flowlot-opencode/lib/python3.12/site-packages/flowio/flowdata.py:451: UserWarning: FCS file 0001.FCS reported incorrect data offset. Attempting to parse data section, but event data should be reviewed before trusting this file.
  warn(warn_msg)
/home/qpb3vt/anaconda3/envs/flowlot-opencode/lib/python3.12/site-packages/flowio/flowdata.py:451: UserWarning: FCS file 0002.FCS reported incorrect data offset. Attempting to parse data section, but event data should be reviewed before trusting this file.
  warn(warn_msg)
/home/qpb3vt/anaconda3/envs/flowlot-opencode/lib/python3.12/site-packages/flowio/flowdata.py:451: UserWarning: FCS file 0003.FCS reported incorrect data offset. Attempting to parse data section, but event data should be reviewed before trusting this file.
  warn(warn_msg)


,dataset,patient_id,tube_id,file,shape,original_count,finite_fraction,markers,original_population_counts,status
0,BLAST110,1,P1,BLAST110_1_P1.fcs,"(212029, 16)",212029,1.0,"[FSC-A, FSC-H, FSC-W, SSC-A, SSC-H, SSC-W, CD7...","{'WBC': 176091, 'Blast': 20335}",OK
1,BLAST110,1,P2,BLAST110_1_P2.fcs,"(212110, 16)",212110,1.0,"[FSC-A, FSC-H, FSC-W, SSC-A, SSC-H, SSC-W, CD1...","{'WBC': 177027, 'Blast': 20539}",OK
2,BLAST110,1,P3,BLAST110_1_P3.fcs,"(213943, 16)",213943,1.0,"[FSC-A, FSC-H, FSC-W, SSC-A, SSC-H, SSC-W, CD3...","{'WBC': 176498, 'Blast': 21382}",OK
3,LAIP29,10_Dx,P1,LAIP29_10_Dx_P1.fcs,"(105869, 16)",105869,1.0,"[FSC-A, FSC-H, FSC-W, SSC-A, SSC-H, SSC-W, CD7...","{'WBC': 103047, 'Blast': 24649, 'LAIP': 18273}",OK
4,LAIP29,10_Dx,P2,LAIP29_10_Dx_P2.fcs,"(100370, 16)",100370,1.0,"[FSC-A, FSC-H, FSC-W, SSC-A, SSC-H, SSC-W, CD1...","{'WBC': 97737, 'Blast': 22943, 'LAIP': 22023}",OK
5,LAIP29,11_Dx,P1,LAIP29_11_Dx_P1.fcs,"(104400, 16)",104400,1.0,"[FSC-A, FSC-H, FSC-W, SSC-A, SSC-H, SSC-W, CD7...","{'WBC': 95972, 'Blast': 9595, 'LAIP': 3944}",OK
6,FLOWCAPII,1,P1,0001.FCS,"(30000, 7)",30000,1.0,"[FS Lin, SS Log, IgG1-FITC, IgG1-PE, CD45-ECD,...",{},OK
7,FLOWCAPII,1,P2,0002.FCS,"(30000, 7)",30000,1.0,"[FS Lin, SS Log, Kappa-FITC, Lambda-PE, CD45-E...",{},OK
8,FLOWCAPII,1,P3,0003.FCS,"(30000, 7)",30000,1.0,"[FS Lin, SS Log, CD7-FITC, CD4-PE, CD45-ECD, C...",{},OK


## 6. Expand and build every requested cell count

`CELL_COUNTS = [1000]` (in the configuration cell) creates one 1000-cell HDF5 level per dataset. With the same seed, smaller levels are nested subsets of larger levels (`500 ⊂ 1000 ⊂ 2000`) whenever enough cells exist. The first write creates the file and later writes append new levels. Existing target groups are protected unless `overwrite=True` is passed deliberately. Build failures are per-configuration: the loop records them and continues. Build failures are per-configuration: the loop records them and continues. Build failures are per-configuration: the loop records them and continues. Build failures are per-configuration: the loop records them and continues.

In [9]:
CONFIGURATIONS = [
    {'dataset': cfg['dataset'], 'manifest': cfg['manifest'], 'stage1': cfg['stage1'], 'cells': cells}
    for cfg in DATASETS
    for cells in CELL_COUNTS
]
pd.DataFrame(CONFIGURATIONS)

,dataset,manifest,stage1,cells
0,BLAST110,/home/qpb3vt/Drive/Naqib/2026_AML_projects_fol...,/home/qpb3vt/Drive/Naqib/2026_AML_projects_fol...,1000
1,LAIP29,/home/qpb3vt/Drive/Naqib/2026_AML_projects_fol...,/home/qpb3vt/Drive/Naqib/2026_AML_projects_fol...,1000
2,FLOWCAPII,/home/qpb3vt/Drive/Naqib/2026_AML_projects_fol...,/home/qpb3vt/Drive/Naqib/2026_AML_projects_fol...,1000


In [ ]:
build_results = []
if RUN_BUILD:
    for cfg in DATASETS:
        if not cfg['manifest'].exists():
            detail = f'manifest missing ({cfg["manifest"]})'
            print(f'SKIPPED {cfg["dataset"]}: {detail}')
            build_results.append({'dataset': cfg['dataset'], 'status': 'skipped', 'cells': None, 'detail': detail})
            continue
        for cell_index, cells in enumerate(CELL_COUNTS):
            print(f"cell_index: {cell_index} cells: {cells}")
            started = time.perf_counter()
            try:
                build_stage1_from_manifest(
                    manifest=cfg['manifest'],
                    output=cfg['stage1'],
                    dataset_name=cfg['dataset'],
                    subsampled_cell_count=cells,
                    seed=SEED,
                    mode='w' if cell_index == 0 else 'a',
                )
                elapsed = time.perf_counter() - started
                print(f"built {cfg['dataset']}/{cells} -> {cfg['stage1'].relative_to(PROJECT_ROOT)} ({elapsed:.1f}s)")
                build_results.append({'dataset': cfg['dataset'], 'status': 'ok', 'cells': cells, 'detail': f'{elapsed:.1f}s'})
            except Exception as error:
                print(f"FAILED {cfg['dataset']}/{cells}: {error!r}")
                build_results.append({'dataset': cfg['dataset'], 'status': 'failed', 'cells': cells, 'detail': repr(error)})
                break
    display(pd.DataFrame(build_results))
else:
    print('No HDF5 written. Set RUN_BUILD=True after all previews and audits pass.')

cell_index: 0 cells: 1000
built BLAST110/1000 -> data/stage1/BLAST110_stage1.h5 (1047.9s)
cell_index: 0 cells: 1000


/home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/flowlot/io/stage1_builder.py:63: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(labels_path)
/home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/flowlot/io/stage1_builder.py:63: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(labels_path)
/home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/flowlot/io/stage1_builder.py:63: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(labels_path)
/home/qpb3vt/Drive/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_opencode/flowlot/io/stage1_builder.py:63: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(l

built LAIP29/1000 -> data/stage1/LAIP29_stage1.h5 (171.8s)
cell_index: 0 cells: 1000


/home/qpb3vt/anaconda3/envs/flowlot-opencode/lib/python3.12/site-packages/flowio/flowdata.py:451: UserWarning: FCS file 0001.FCS reported incorrect data offset. Attempting to parse data section, but event data should be reviewed before trusting this file.
  warn(warn_msg)
/home/qpb3vt/anaconda3/envs/flowlot-opencode/lib/python3.12/site-packages/flowio/flowdata.py:451: UserWarning: FCS file 0002.FCS reported incorrect data offset. Attempting to parse data section, but event data should be reviewed before trusting this file.
  warn(warn_msg)
/home/qpb3vt/anaconda3/envs/flowlot-opencode/lib/python3.12/site-packages/flowio/flowdata.py:451: UserWarning: FCS file 0003.FCS reported incorrect data offset. Attempting to parse data section, but event data should be reviewed before trusting this file.
  warn(warn_msg)
/home/qpb3vt/anaconda3/envs/flowlot-opencode/lib/python3.12/site-packages/flowio/flowdata.py:451: UserWarning: FCS file 0004.FCS reported incorrect data offset. Attempting to parse 

## 7. Verify the Stage 1 file and summarize every split

This checks the stored schema and reports patient/tube coverage, event counts, and the exact sampled/original WBC, Blast, and LAIP values. A stored count may be below the requested level when fewer annotated events are available.

In [ ]:
for cfg in DATASETS:
    stage1 = cfg['stage1']
    if not stage1.exists():
        print(f"No Stage 1 file yet: {stage1.relative_to(PROJECT_ROOT)}")
        continue
    stage1_inventory, stage1_issues = audit_stage1(stage1)
    print(f"Stage 1 {stage1.name}: rows={len(stage1_inventory)}, issues={len(stage1_issues)}")
    if not stage1_issues.empty:
        display(stage1_issues)
    statistics, population_statistics = [], []
    with h5py.File(stage1, 'r') as handle:
        for dataset_name in handle:
            for cell_level in handle[dataset_name]:
                for sample_key, sample in handle[dataset_name][cell_level].items():
                    for tube_id, tube in sample.items():
                        matrix = tube['raw_cell_matrix']
                        statistics.append({
                            'dataset': dataset_name, 'cell_level': cell_level,
                            'patient_id': tube.attrs.get('patient_id', sample_key),
                            'label': tube.attrs.get('label'), 'tube_id': tube_id,
                            'stored_cells': matrix.shape[0], 'markers': matrix.shape[1],
                            'original_count': int(tube.attrs['counts']),
                        })
                        if 'population_counts' in tube:
                            counts = tube['population_counts']
                            names = [value.decode() for value in counts.attrs['population_names']]
                            metrics = [value.decode() for value in counts.attrs['metric_names']]
                            for pop_index, population in enumerate(names):
                                population_statistics.append({
                                    'dataset': dataset_name, 'cell_level': cell_level,
                                    'patient_id': tube.attrs.get('patient_id', sample_key),
                                    'tube_id': tube_id, 'population': population,
                                    **dict(zip(metrics, counts[pop_index])),
                                })
    stats = pd.DataFrame(statistics)
    display(stats.head())
    display(stats.groupby(['dataset', 'cell_level', 'tube_id']).agg(
        patients=('patient_id', 'nunique'),
        labels=('label', 'nunique'),
        stored_min=('stored_cells', 'min'),
        stored_median=('stored_cells', 'median'),
        original_min=('original_count', 'min'),
        original_max=('original_count', 'max'),
    ).reset_index())
    if population_statistics:
        display(pd.DataFrame(population_statistics))

In [ ]:
# -------------------- Run summary (successful vs failed per dataset) --------------------
summary_rows = []
for cfg in DATASETS:
    manifest_ok = cfg['manifest'].exists()
    stage1_ok = cfg['stage1'].exists()
    if manifest_ok and stage1_ok:
        status = 'ok'
    elif manifest_ok or stage1_ok:
        status = 'partial'
    else:
        status = 'failed'
    summary_rows.append({
        'dataset': cfg['dataset'],
        'manifest': 'ok' if manifest_ok else 'missing',
        'stage1': 'ok' if stage1_ok else 'missing',
        'status': status,
        'output': str(cfg['stage1'].relative_to(PROJECT_ROOT)),
    })
summary = pd.DataFrame(summary_rows)
display(summary)
print('Successful datasets:', summary['status'].eq('ok').sum(), 'of', len(summary))

## Troubleshooting

- **FCS import error**: install the `fcs` extra and restart the kernel.
- **Unmatched files**: inspect relative paths in the discovery cell and adjust `filename_pattern`; do not silently relabel them.
- **Missing labels**: ensure IDs in the label CSV exactly match IDs captured from filenames (including leading zeros).
- **Marker-count mismatch**: the declared marker list must have one entry per matrix column.
- **Mixed panels**: map marker names separately under each `tube_id`.
- **Need to rebuild**: delete/move the output deliberately or use a new output name; safeguards prevent accidental replacement.